# Backpropagation Derivation

In [1]:

"""Forward propagation for a two-layer neural network."""

import numpy as np


def sigmoid(z):
    """Calculate the sigmoid activation function."""
    return 1 / (1 + np.exp(-z))


def forward():
    """Calculate forward propagation using fixed mission values."""
    x = np.array([1.0, 0.0])

    W1 = np.array([
        [0.1, 0.2],
        [0.3, 0.4]
    ])

    b1 = np.array([0.0, 0.0])
    W2 = np.array([0.5, 0.6])
    b2 = 0.0

    z1 = W1 @ x + b1
    a1 = sigmoid(z1)

    z2 = W2 @ a1 + b2
    y_pred = sigmoid(z2)

    print("z1:", np.round(z1, 4), "shape:", z1.shape)
    print("a1:", np.round(a1, 4), "shape:", a1.shape)
    print("z2:", round(float(z2), 4), "shape:", np.shape(z2))
    print("y_pred:", round(float(y_pred), 4),
          "shape:", np.shape(y_pred))


if __name__ == "__main__":
    forward()


z1: [0.1 0.3] shape: (2,)
a1: [0.525  0.5744] shape: (2,)
z2: 0.6072 shape: ()
y_pred: 0.6473 shape: ()


In [2]:

# Binary Cross-Entropy Loss

y_true = 1.0

def binary_cross_entropy(y_true, y_pred):
    """Calculate binary cross-entropy loss."""
    return -(
        y_true * np.log(y_pred)
        + (1 - y_true) * np.log(1 - y_pred)
    )

# Reuse the forward propagation function's calculations.
x = np.array([1.0, 0.0])
W1 = np.array([[0.1, 0.2], [0.3, 0.4]])
b1 = np.array([0.0, 0.0])
W2 = np.array([0.5, 0.6])
b2 = 0.0

z1 = W1 @ x + b1
a1 = sigmoid(z1)
z2 = W2 @ a1 + b2
y_pred = sigmoid(z2)

loss = binary_cross_entropy(y_true, y_pred)

print(f"y_true: {y_true}")
print(f"y_pred: {y_pred:.6f}")
print(f"BCE Loss: {loss:.6f}")


y_true: 1.0
y_pred: 0.647292
BCE Loss: 0.434958


In [5]:

# Backpropagation using the chain rule

def sigmoid_derivative(a):
    """Calculate sigmoid derivative from its output."""
    return a * (1 - a)


# 1. BCE loss derivative with respect to prediction
dL_dy_pred = (
    -y_true / y_pred
    + (1 - y_true) / (1 - y_pred)
)

# 2. Output layer gradient
dL_dz2 = y_pred - y_true

# 3. Gradient for W2
dL_dW2 = dL_dz2 * a1

# 4. Gradient passed to hidden layer
dL_da1 = dL_dz2 * W2

# 5. Hidden layer gradient
dL_dz1 = dL_da1 * sigmoid_derivative(a1)

# 6. Gradient for W1
dL_dW1 = np.outer(dL_dz1, x)


print("Backpropagation Results")
print("dL/dy_pred:", round(float(dL_dy_pred), 4))
print("dL/dz2:", round(float(dL_dz2), 4))
print("dL/dW2:", np.round(dL_dW2, 4))
print("dL/da1:", np.round(dL_da1, 4))
print("dL/dz1:", np.round(dL_dz1, 4))
print("dL/dW1:\n", np.round(dL_dW1, 4))

print("\nGradient Shapes")
for name, value in [
    ("dL/dy_pred", dL_dy_pred),
    ("dL/dz2", dL_dz2),
    ("dL/dW2", dL_dW2),
    ("dL/da1", dL_da1),
    ("dL/dz1", dL_dz1),
    ("dL/dW1", dL_dW1)
]:
    print(name, np.shape(value))


Backpropagation Results
dL/dy_pred: -1.5449
dL/dz2: -0.3527
dL/dW2: [-0.1852 -0.2026]
dL/da1: [-0.1764 -0.2116]
dL/dz1: [-0.044  -0.0517]
dL/dW1:
 [[-0.044  -0.    ]
 [-0.0517 -0.    ]]

Gradient Shapes
dL/dy_pred ()
dL/dz2 ()
dL/dW2 (2,)
dL/da1 (2,)
dL/dz1 (2,)
dL/dW1 (2, 2)



## Backpropagation Derivation

### 1. Network Structure

Input (2) → Hidden (2) → Output (1)

- x: (2,) = [1, 0]
- W1: (2, 2) = [[0.1, 0.2], [0.3, 0.4]]
- b1: (2,) = [0, 0]
- W2: (2,) = [0.5, 0.6]
- b2: scalar = 0
- y_true: scalar = 1

### 2. Forward Propagation

$$z_1 = W_1x+b_1 = [0.1,0.3]$$

$$a_1 = \sigma(z_1) = [0.5250,0.5744]$$

$$z_2 = W_2a_1+b_2 \approx 0.6072$$

$$\hat y = \sigma(z_2) \approx 0.6473$$

$$L=-\log(\hat y)\approx0.434958$$

### 3. Backpropagation Using Chain Rule

**Step 1: BCE derivative (scalar)**

$$
\frac{\partial L}{\partial \hat y}
=-\frac{y}{\hat y}+\frac{1-y}{1-\hat y}
\approx-1.5449
$$

**Step 2: Output layer gradient (scalar)**

$$
\frac{\partial L}{\partial z_2}
=\frac{\partial L}{\partial \hat y}
\frac{\partial \hat y}{\partial z_2}
=\hat y-y
\approx-0.3527
$$

**Step 3: W2 gradient (2,)**

$$
\frac{\partial L}{\partial W_2}
=\frac{\partial L}{\partial z_2}a_1
\approx[-0.1852,-0.2026]
$$

**Step 4: Hidden activation gradient (2,)**

$$
\frac{\partial L}{\partial a_1}
=\frac{\partial L}{\partial z_2}W_2
\approx[-0.1764,-0.2116]
$$

**Step 5: Hidden pre-activation gradient (2,)**

$$
\frac{\partial L}{\partial z_1}
=\frac{\partial L}{\partial a_1}
\odot a_1(1-a_1)
\approx[-0.0440,-0.0517]
$$

**Step 6: W1 gradient (2,2)**

$$
\frac{\partial L}{\partial W_1}
=\frac{\partial L}{\partial z_1}x^T
\approx
\begin{bmatrix}
-0.0440&0\\
-0.0517&0
\end{bmatrix}
$$

### 4. Validation

NumPy calculations agree with the independently
derived chain-rule results to four decimal places.

Note: The mission's sample shows -0.0513,
but the given weights produce approximately -0.0517.


In [6]:

# Compare hand calculations with NumPy results

# Expected values from the chain-rule derivation
hand_calculated = {
    "dL/dy_pred": -1.5449,
    "dL/dz2": -0.3527,
    "dL/dW2": np.array([-0.1852, -0.2026]),
    "dL/da1": np.array([-0.1764, -0.2116]),
    "dL/dz1": np.array([-0.0440, -0.0517]),
    "dL/dW1": np.array([
        [-0.0440, 0.0],
        [-0.0517, 0.0]
    ])
}

numpy_results = {
    "dL/dy_pred": dL_dy_pred,
    "dL/dz2": dL_dz2,
    "dL/dW2": dL_dW2,
    "dL/da1": dL_da1,
    "dL/dz1": dL_dz1,
    "dL/dW1": dL_dW1
}

all_passed = True

for name, expected in hand_calculated.items():
    actual = numpy_results[name]

    passed = np.array_equal(
        np.round(expected, 4),
        np.round(actual, 4)
    )

    print(f"{name}: {'PASS' if passed else 'FAIL'}")

    if not passed:
        all_passed = False

print("\nFinal Validation:",
      "PASS" if all_passed else "FAIL")


dL/dy_pred: PASS
dL/dz2: PASS
dL/dW2: PASS
dL/da1: PASS
dL/dz1: PASS
dL/dW1: PASS

Final Validation: PASS
